# APAL数据集拓扑复核（2026-10-01）

## 结果摘要

四份数据均为有效DAG；虚拟节点依赖不可忽略。旧283参考的18道初步候选中，13道无同站后继，5道各有1道。大规模数据的无入边Sub含义待确认，统计按所读加载器规则解释。原始输入不改动。

## 背景与方法

### 关键假设

物理节点按类型2判断；TaskID按零基原始行号匹配旧283排程，生成版本未独立验证。标准工时与实际延迟分开。空Sub前序不人工补顺序。完整来源及局限见同名解析报告和source_manifest.json。

依赖：Python、pandas、numpy。输出为小表；暂不绘制尚待工艺语义核准的全图分布结论。

### 方法代码 1

可复核实现；不输出原始记录。

In [1]:
"""APAL topology audit. Inputs stay unchanged; raw rows are never printed.
Run: python analyze_topology.py --input-dir ../upload --output-dir .
Only pandas/numpy are needed. Generated metrics concern static precedence,
not realized delays, future disturbances, or optimized schedules.
"""
from __future__ import annotations
import argparse, ast, collections, hashlib, heapq, json, re
from pathlib import Path
import numpy as np
import pandas as pd
LOADER_BLOB = '0ca1c0f0208e484bc3109f7c7377286dd5c1a911'

### 方法代码 2

可复核实现；不输出原始记录。

In [2]:
def tokens(value):
    if pd.isna(value):
        return []
    result = []
    for raw in re.split('[,，;；、]', str(value)):
        t = raw.strip()
        if t.endswith('.0'):
            t = t[:-2]
        if t.lower() not in {'', '0', 'nan', 'none'}:
            result.append(t)
    return result

def bit_ids(bits):
    while bits:
        x = bits & -bits
        yield (x.bit_length() - 1)
        bits ^= x

def topology(n, edges):
    adj = [set() for _ in range(n)]
    pred = [set() for _ in range(n)]
    for u, v in edges:
        adj[u].add(v)
        pred[v].add(u)
    deg = [len(x) for x in pred]
    heap = [i for i in range(n) if not deg[i]]
    heapq.heapify(heap)
    order = []
    while heap:
        u = heapq.heappop(heap)
        order.append(u)
        for v in sorted(adj[u]):
            deg[v] -= 1
            if deg[v] == 0:
                heapq.heappush(heap, v)
    if len(order) != n:
        raise ValueError('Cycle in precedence graph')
    return (adj, pred, order)

def descendant_bits(adj, order, physical):
    bits = [0] * len(adj)
    for u in reversed(order):
        for v in adj[u]:
            bits[u] |= bits[v] | (1 << v if v in physical else 0)
    return bits

def stats(values):
    a = np.asarray(values, dtype=float)
    return {k: float(v) for k, v in zip(['min', 'q25', 'median', 'q75', 'p90', 'max', 'mean'], [a.min(), *np.quantile(a, [0.25, 0.5, 0.75, 0.9]), a.max(), a.mean()])}

### 方法代码 3

可复核实现；不输出原始记录。

In [3]:
def build_tagged(df):
    ids = df.task_id.tolist()
    imap = {s: i for i, s in enumerate(ids)}
    rules = {r: set() for r in ['A', 'B', 'C', 'D', 'E']}
    roots = []
    root_subs = {}
    groups = {}
    parent_sub = {}
    parent_root = {}
    current_root = current_sub = None
    for i, tid in enumerate(ids):
        if '-' not in tid and tid.isalpha():
            current_root = i
            current_sub = None
            roots.append(i)
            root_subs[i] = []
        elif '-' in tid and tid.split('-')[0].isalpha() and tid.split('-')[1].isdigit():
            current_sub = i
            groups[i] = []
            if current_root is not None:
                root_subs[current_root].append(i)
            parent_root[i] = current_root
        else:
            if current_sub is not None:
                groups[current_sub].append(i)
            parent_sub[i] = current_sub
            parent_root[i] = current_root
    ss = {s: [] for s in groups}
    dangling = []
    for v, row in df.iterrows():
        for p in tokens(row.predecessors):
            if p not in imap:
                dangling.append({'row': int(v) + 2, 'task': row.task_id, 'pred': p})
                continue
            u = imap[p]
            rules['D'].add((u, v))
            if v in ss and u in ss and (v not in ss[u]):
                ss[u].append(v)
    if dangling:
        raise ValueError('Dangling predecessors: ' + str(dangling[:5]))
    for r in roots:
        if root_subs[r]:
            rules['E'].add((r, root_subs[r][0]))
    for s, ts in groups.items():
        rules['A'].update(((s, t) for t in ts))
    for ri, r in enumerate(roots):
        for s in root_subs[r]:
            if ss[s]:
                rules['B'].update(((t, v) for t in groups[s] or [s] for v in ss[s]))
            elif ri + 1 < len(roots):
                rules['C'].update(((t, roots[ri + 1]) for t in groups[s] or [s]))
    edges = set().union(*rules.values())
    return (edges, rules, roots, root_subs, groups, parent_sub, parent_root, ss)

def cross_check_loader(df, loader_path, edges):
    """Execute unchanged graph-building AST only, excluding feature/tensor dependencies.
    No synthetic fields are supplied; this does not claim production-loader execution.
    """
    tree = ast.parse(loader_path.read_text())
    f = next((x for x in tree.body if isinstance(x, ast.FunctionDef) and x.name == 'load_data'))
    start = next((i for i, x in enumerate(f.body) if isinstance(x, ast.Assign) and any((isinstance(t, ast.Name) and t.id == 'edges' for t in x.targets))))
    end = next((i for i in range(start + 1, len(f.body)) if isinstance(f.body[i], ast.Assign) and any((isinstance(t, ast.Name) and t.id == 'edges' for t in f.body[i].targets)) and isinstance(f.body[i].value, ast.Call)))
    statements = f.body[start:end + 1]
    env = {'df': df, 'id_map': dict(zip(df.task_id, df.internal_id)), 'np': np, 'pd': pd, 're': re, 'Any': object}
    parser = next((x for x in tree.body if isinstance(x, ast.FunctionDef) and x.name == '_parse_predecessor_tokens'))
    exec(compile(ast.Module(body=[parser], type_ignores=[]), 'loader_parser_subset', 'exec'), env)
    exec(compile(ast.Module(body=statements, type_ignores=[]), 'loader_graph_subset', 'exec'), env)
    assert edges == set(env['edges']), 'Tagged parser differs from loader graph subset'
    return ast.unparse(ast.Module(body=statements, type_ignores=[]))

### 方法代码 4

可复核实现；不输出原始记录。

In [4]:
def analyze(path, out, loader=None):
    raw = pd.read_csv(path)
    df = raw.rename(columns={'AO号': 'task_id', '类型': 'node_type', '工种': 'skill', '专业编码': 'profession', '紧前工序AO号': 'predecessors', '需求人数': 'workers', '加工时间/h': 'duration', '限定站位': 'fixed'})
    df['task_id'] = df.task_id.astype(str).str.strip()
    df['internal_id'] = range(len(df))
    assert not df.task_id.duplicated().any(), 'Duplicate AO identifiers'
    assert df.node_type.isin([1, 2]).all()
    assert (df.duration >= 0).all()
    phys = set(df.index[df.node_type.eq(2)])
    virtual = set(df.index) - phys
    assert df.loc[list(phys), 'skill'].isin(range(5)).all()
    assert df.loc[list(virtual), 'skill'].eq(-1).all()
    assert df.loc[list(phys), 'workers'].ge(1).all()
    edges, rules, roots, root_subs, groups, ps, pr, ss = build_tagged(df)
    hierarchy = set(roots) | set(groups)
    assert virtual == hierarchy, 'Hierarchy versus node type mismatch'
    assert set(ps) == phys and all((ps[i] is not None for i in phys)), 'Ungrouped physical tasks'
    loader_match = False
    if loader and loader.exists():
        cross_check_loader(df, loader, edges)
        loader_match = True
    adj, pred, order = topology(len(df), edges)
    desc = descendant_bits(adj, order, phys)
    first = [0] * len(df)
    for u in reversed(order):
        if u in phys:
            first[u] = 1 << u
        else:
            for v in adj[u]:
                first[u] |= first[v]
    pedges = set()
    for u in phys:
        endpoints = 0
        for v in adj[u]:
            endpoints |= first[v]
        pedges.update(((u, v) for v in bit_ids(endpoints)))
    padj, ppred, porder = topology(len(df), pedges)
    pdesc = descendant_bits(padj, porder, phys)
    assert all((pdesc[i] == desc[i] for i in phys)), 'Virtual elimination changed physical reachability'
    dedges = {(u, v) for u, v in rules['D'] if u in phys and v in phys}
    dadj, _, dorder = topology(len(df), dedges)
    ddesc = descendant_bits(dadj, dorder, phys)
    depth = [0] * len(df)
    eft = [0.0] * len(df)
    for u in order:
        depth[u] = max((depth[p] for p in pred[u]), default=0) + (u in phys)
        eft[u] = max((eft[p] for p in pred[u]), default=0) + (float(df.loc[u, 'duration']) if u in phys else 0)
    duration = df.duration.to_numpy(float)
    cache = {}

    def work(b):
        if b not in cache:
            cache[b] = sum((duration[i] for i in bit_ids(b)))
        return cache[b]
    records = []
    for i in sorted(phys):
        records.append({'dataset': path.stem, 'TaskID': i, 'AO': df.loc[i, 'task_id'], 'Root': df.loc[pr[i], 'task_id'], 'Sub': df.loc[ps[i], 'task_id'], 'skill': int(df.loc[i, 'skill']), 'workers': int(df.loc[i, 'workers']), 'standard_hours': duration[i], 'fixed_station': None if pd.isna(df.loc[i, 'fixed']) else int(df.loc[i, 'fixed']), 'physical_next_count': len(padj[i]), 'physical_previous_count': len(ppred[i]), 'all_physical_descendants': desc[i].bit_count(), 'descendant_standard_hours': work(desc[i]), 'explicit_only_descendants': ddesc[i].bit_count(), 'physical_path_depth': depth[i]})
    metrics = pd.DataFrame(records)
    group_records = []
    for s, ts in groups.items():
        group_records.append({'dataset': path.stem, 'Root': df.loc[pr[s], 'task_id'], 'Sub': df.loc[s, 'task_id'], 'physical_tasks': len(ts), 'standard_hours': float(df.loc[ts, 'duration'].sum()), 'sub_successors': [df.loc[t, 'task_id'] for t in ss[s]], 'sub_predecessors': [p for p in tokens(df.loc[s, 'predecessors']) if p in set(df.loc[list(groups), 'task_id'])], 'explicit_predecessor_AOs': tokens(df.loc[s, 'predecessors']), 'full_graph_in_degree': len(pred[s]), 'reachable_from_own_root': bool(desc[pr[s]] & sum((1 << t for t in ts))) if ts else None})
    total_pairs = sum((desc[i].bit_count() for i in phys))
    explicit_pairs = sum((ddesc[i].bit_count() for i in phys))
    unseen = set(range(len(df)))
    components = []
    while unseen:
        seed = next(iter(unseen))
        todo = [seed]
        reached = {seed}
        while todo:
            u = todo.pop()
            for v in adj[u] | pred[u]:
                if v not in reached:
                    reached.add(v)
                    todo.append(v)
        unseen -= reached
        components.append(len(reached))
    summary = {'dataset': path.stem, 'input_sha256': hashlib.sha256(path.read_bytes()).hexdigest(), 'rows': len(df), 'physical': len(phys), 'virtual': len(virtual), 'roots': len(roots), 'subs': len(groups), 'full_edges': len(edges), 'edge_rules': {k: len(v) for k, v in rules.items()}, 'physical_projected_edges': len(pedges), 'explicit_physical_edges': len(dedges), 'physical_precedence_pairs': total_pairs, 'explicit_only_precedence_pairs': explicit_pairs, 'lost_pair_share_if_explicit_only': (total_pairs - explicit_pairs) / total_pairs, 'weak_components': len(components), 'is_DAG': True, 'loader_graph_subset_matches': loader_match, 'physical_projection_reachability_matches': True, 'fixed_tasks': metrics[metrics.fixed_station.notna()][['TaskID', 'AO', 'fixed_station']].to_dict('records'), 'zero_duration_physical': int(df.loc[list(phys), 'duration'].eq(0).sum()), 'skills': df.loc[list(phys), 'skill'].value_counts().sort_index().to_dict(), 'descendant_count': stats(metrics.all_physical_descendants), 'descendant_standard_hours': stats(metrics.descendant_standard_hours), 'physical_chain_max_nodes': max(depth), 'standard_hour_path_max': max(eft), 'sub_forks': sum((len(v) > 1 for v in ss.values())), 'sub_joins': sum((len(g['sub_predecessors']) > 1 for g in group_records)), 'group_sizes': stats([len(v) for v in groups.values()]), 'issues': []}
    if any(df.loc[list(virtual), 'duration'].ne(0)) or any(df.loc[list(virtual), 'workers'].ne(0)):
        summary['issues'].append('Virtual nodes carry nonzero workload')
    summary['unanchored_subs'] = [df.loc[s, 'task_id'] for s in groups if not pred[s]]
    summary['physical_not_reachable_from_own_root'] = sum((not desc[pr[i]] & 1 << i for i in phys))
    summary['root_coverage'] = [{'root': df.loc[r, 'task_id'], 'physical_tasks': sum((pr[i] == r for i in phys)), 'physical_not_reachable': sum((pr[i] == r and (not desc[r] & 1 << i) for i in phys))} for r in roots]
    if summary['unanchored_subs']:
        summary['issues'].append('Source Sub groups without incoming edges: blank precedence semantics need confirmation; do not automatically impose sequence')
    metrics.to_csv(out / f'{path.stem}_physical_task_metrics.csv', index=False, encoding='utf-8-sig')
    return (summary, metrics, group_records, {'df': df, 'phys': phys, 'desc': desc, 'edges': edges, 'pedges': pedges, 'adj': adj, 'pred': pred})

### 方法代码 5

可复核实现；不输出原始记录。

In [5]:
def analyze_schedule(path, state, metrics, out):
    s = pd.read_csv(path)
    df = state['df']
    phys = state['phys']
    desc = state['desc']
    assert not s.TaskID.duplicated().any()
    assert set(s.TaskID) == set(df.index), 'Schedule IDs do not cover 283 row IDs'
    s = s.set_index('TaskID').sort_index()
    p = s.loc[sorted(phys)]
    assert p.StationID.between(1, 5).all()
    mapping = {int(i): int(v) for i, v in p.StationID.items()}
    fixed = {int(i): int(v) for i, v in df.fixed.dropna().items() if i in phys}
    station_viol = [(u, v) for u, v in state['pedges'] if mapping[u] > mapping[v]]
    time_viol = [(u, v) for u, v in state['edges'] if float(s.loc[u, 'End']) > float(s.loc[v, 'Start']) + 1e-07]
    fixed_viol = [i for i, v in fixed.items() if mapping[i] != v]
    workers = collections.defaultdict(set)
    intervals = collections.defaultdict(list)
    team_count_viol = []
    for i, row in p.iterrows():
        team = ast.literal_eval(row.Team)
        if len(team) != int(df.loc[i, 'workers']) or len(team) != len(set(team)):
            team_count_viol.append(int(i))
        for w in team:
            workers[w].add(int(row.StationID))
            intervals[w].append((float(row.Start), float(row.End), int(i)))
    overlaps = []
    for w, ivs in intervals.items():
        end = -float('inf')
        previous = None
        for a, b, i in sorted(ivs):
            if a < end - 1e-07:
                overlaps.append((int(w), previous, i))
            if b > end:
                end = b
                previous = i
    merged = metrics.copy()
    merged['old_reference_station'] = merged.TaskID.map(mapping)
    rows = []
    for row in merged.itertuples():
        i = int(row.TaskID)
        m = mapping[i]
        same = [j for j in bit_ids(desc[i]) if mapping[j] == m]
        blockers = [j for j in bit_ids(desc[i]) if j in fixed and fixed[j] < m + 1]
        rows.append({'TaskID': i, 'same_station_descendants': len(same), 'same_station_descendant_standard_hours': float(df.loc[same, 'duration'].sum()), 'same_station_descendant_old_duration_sum': float(s.loc[same, 'Duration'].sum()), 'candidate_by_unfixed_and_not_last': i not in fixed and m < 5, 'fixed_descendant_blockers_for_one_station_deferral': len(blockers), 'same_station_descendant_AOs': ';'.join(df.loc[same, 'task_id'])})
    merged = merged.merge(pd.DataFrame(rows), on='TaskID', validate='one_to_one')
    merged.to_csv(out / '283_old_reference_station_cascade.csv', index=False, encoding='utf-8-sig')
    candidate = merged[merged.candidate_by_unfixed_and_not_last]
    summary = {'source_file': path.name, 'input_sha256': hashlib.sha256(path.read_bytes()).hexdigest(), 'matched_dataset': '283', 'mapping_assumption': 'TaskID is zero-based original CSV row index; no AO field in old schedule, so exact generating revision unverified', 'rows': len(s), 'physical_tasks': len(p), 'virtual_station_zero': int(s.loc[sorted(set(df.index) - phys), 'StationID'].eq(0).sum()), 'physical_station_counts': p.StationID.value_counts().sort_index().to_dict(), 'last_station_share': float(p.StationID.eq(5).mean()), 'physical_station_precedence_violations': len(station_viol), 'full_edge_numeric_time_violations': len(time_viol), 'fixed_station_violations': len(fixed_viol), 'team_size_violations': len(team_count_viol), 'workers_used': len(workers), 'multi_station_workers': sum((len(v) > 1 for v in workers.values())), 'worker_time_overlap_pairs': len(overlaps), 'duration_consistency_max_error': float((p.End - p.Start - p.Duration).abs().max()), 'candidates_unfixed_nonlast': len(candidate), 'candidate_same_station_descendants': stats(candidate.same_station_descendants), 'candidate_zero_same_station_descendants': int(candidate.same_station_descendants.eq(0).sum()), 'candidate_fixed_descendant_blocked': int(candidate.fixed_descendant_blockers_for_one_station_deferral.gt(0).sum()), 'max_same_station_among_all_tasks': int(merged.same_station_descendants.max()), 'largest_candidates': candidate.sort_values(['same_station_descendants', 'all_physical_descendants'], ascending=False).head(6)[['TaskID', 'AO', 'old_reference_station', 'all_physical_descendants', 'same_station_descendants', 'same_station_descendant_AOs']].to_dict('records'), 'limitations': ['Old template, not Work3 real multi-aircraft trajectory', 'Future station skills/efficiencies/calendars absent; no full deferral feasibility certification', 'Single highly unbalanced reference cannot represent all schedules', 'Static initial sets include tasks that would later be completed; actual unfinished cascade is state-dependent']}
    return (summary, merged)

## 数据

将原始四份CSV和旧排程文本放在INPUT_DIR。可选加载器快照必须是报告注明的版本，路径不存在时仅使用已记录的构图规则，逐边比对标志为False。

In [6]:
INPUT_DIR = Path('../upload')
OUTPUT_DIR = Path('notebook_output')
LOADER_SNAPSHOT = Path('../code_check_oct/data_loader.py')
OUTPUT_DIR.mkdir(exist_ok=True)


## 结果

### 四份拓扑汇总

下界和后继范围按当前规则统计；无入边Sub是需要解释的现象，不自动判为错误。

In [7]:
results=[]
frames=[]
states={}
for label in ['283','680','2338','3182']:
    summary,frame,group,state=analyze(INPUT_DIR/(label+'.csv'),OUTPUT_DIR,LOADER_SNAPSHOT)
    results.append(summary)
    frames.append(frame)
    states[label]=state
summary_table=pd.DataFrame([{'数据':x['dataset'],'物理工序':x['physical'],'虚拟节点':x['virtual'],'最大物理后继':int(x['descendant_count']['max']),'无入边Sub':len(x['unanchored_subs']),'匹配加载器构图':x['loader_graph_subset_matches']} for x in results])
display(summary_table)


数据,物理工序,虚拟节点,最大物理后继,无入边Sub,匹配加载器构图
283,283,7,256,0,True
680,680,35,278,12,True
2338,2338,64,450,27,True
3182,3182,117,966,45,True


### 旧283参考的同站连带

只检查未限定站位且不在末站的初步候选。未验证目标站技能或动态提交决定。

In [8]:
old_summary,cascade=analyze_schedule(INPUT_DIR/'粘贴的文本 (1).txt',states['283'],frames[0],OUTPUT_DIR)
counts=cascade[cascade.candidate_by_unfixed_and_not_last].same_station_descendants.value_counts().sort_index()
display(pd.DataFrame({'同站后继数量':counts.index,'初步候选数量':counts.values}))
display(cascade[cascade.TaskID.eq(20)][['AO','old_reference_station','all_physical_descendants','same_station_descendants','same_station_descendant_AOs']])
assert int(counts.sum())==18
assert int(counts.loc[0])==13 and int(counts.loc[1])==5


同站后继数量,初步候选数量
0,13
1,5


AO,old_reference_station,all_physical_descendants,same_station_descendants,same_station_descendant_AOs
AJCP00-0150,3,256,1,AJCP00-0160


## 后续讨论

先解释B、C、D、F等工艺阶段中无紧前Sub的含义，再重算与确认联动策略。现有旧排程92.23%的物理任务在末站，不能代表更均衡排程的迁移规模。完整指标导出为CSV，可复查但无需逐行进入对话。